# Đánh giá PaddleOCR trên golden set tổng hợp

Chạy PaddleOCR lên các giấy tờ do `src/data-golden-set-generator` sinh ra, so với `ground_truth.json` để xem:

1. **End-to-end** (detection + recognition trên cả trang): mỗi trường có đọc ra đúng không, theo mức nhiễu `clean → light → medium → heavy`.
2. **Recognition-only** (cắt đúng bbox ground truth rồi nhận dạng): tách lỗi do model nhận dạng với lỗi do detection.
3. **Ký tự hay sai**: tiếng Việt có dấu chồng (ọ, ữ, ệ…) là điểm yếu cần xem kỹ.

**Chỉ số:**

- **CER** = khoảng cách Levenshtein / số ký tự ground truth. End-to-end dùng khớp bán toàn cục: lấy chuỗi con khớp nhất trong các dòng OCR chồng lên bbox trường, vì dòng OCR thường dính cả nhãn (`Họ và tên: ...`).
- **WER**: như CER nhưng tính theo từ. Sai một dấu là sai cả từ.
- **Field Accuracy**: tỷ lệ trường đọc đúng hoàn toàn (CER = 0).
- **Document Accuracy**: tỷ lệ giấy tờ (ở một mức nhiễu) có mọi trường đọc đúng.
- **CER bỏ dấu**: CER sau khi bỏ dấu và chữ hoa. Chênh lệch giữa hai CER là phần lỗi do dấu.
- **Detection recall**: tỷ lệ trường có ít nhất một dòng OCR chồng lên bbox.

Trường `checkbox` bị loại, OCR không đọc ô tích.

> Đây là dữ liệu tổng hợp, metric sẽ đẹp hơn hồ sơ thật. Dùng để so model và mức nhiễu, không dùng làm con số cuối.

## 0. Cài đặt (chỉ cần trên Colab)

Máy local dùng `jupyters/.venv` (xem `README.md`). Trên Colab, chọn runtime GPU rồi chạy cell dưới.

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install -q paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
    %pip install -q paddleocr==3.7.0 rapidfuzz

## 1. Cấu hình

Dữ liệu để trong thư mục `data/` cạnh notebook, là kết quả của generator (`generated/`). Notebook quét đệ quy `data/`, lấy đúng các loại ảnh generator sinh ra:

| File | Ý nghĩa | Dùng |
|---|---|---|
| `<doc-id>_p<N>.png` | Trang N bản sạch | Có |
| `<doc-id>_p<N>_<light\|medium\|heavy>.jpg` | Trang N đã thêm nhiễu | Có |
| `<doc-id>_p<N>[_<level>]_bbox.jpg` | Ảnh debug vẽ bbox | Không |
| `<doc-id>.pdf`, `<doc-id>.html` | Bản render gốc | Không |
| `ground_truth.json` | Nhãn của cả hồ sơ, cùng thư mục với ảnh | Chấm điểm |

- **Local:** `ln -s ../src/data-golden-set-generator/generated data`
- **Colab:** upload file zip thư mục `generated/` rồi giải nén thành `/content/data`.

In [ ]:
import os
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

import json, re, time, unicodedata
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rapidfuzz.distance import Levenshtein

DATA_DIR = Path("data")                  # tương đối với thư mục notebook (trên Colab là /content)
CACHE_DIR = Path("outputs/paddleocr")    # kết quả OCR, chạy lại không tốn thời gian
# Tên ảnh do generator sinh: <doc-id>_p<N>.png (sạch), <doc-id>_p<N>_<level>.jpg (nhiễu). Ảnh *_bbox.jpg không khớp
IMAGE_PATTERN = re.compile(r"^(?P<doc_id>.+)_p(?P<page>\d+)(?:\.png|_(?P<level>light|medium|heavy)\.jpg)$")

MAX_IMAGES = 24                          # None = tất cả. CPU ~25 giây/ảnh/model, GPU nhanh hơn nhiều
LEVELS = ["clean", "light", "medium", "heavy"]
DET_MODEL = "PP-OCRv6_medium_det"        # giữ cố định để so riêng model nhận dạng
REC_MODELS = [
    "PP-OCRv6_medium_rec",               # model PaddleOCR chọn cho lang="vi"
    "latin_PP-OCRv5_mobile_rec",
    "PP-OCRv5_server_rec",
]
import paddle
if paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0:
    PADDLE_KWARGS = dict(device="gpu:0")
else:
    # Paddle CPU bản 3.x lỗi oneDNN (ConvertPirAttribute2RuntimeAttribute) với model v6, nên tắt
    PADDLE_KWARGS = dict(device="cpu", enable_mkldnn=False)
print(PADDLE_KWARGS)

pd.set_option("display.max_colwidth", 80)

In [ ]:
# Quy ước biểu đồ: màu categorical cố định theo model (không theo thứ hạng),
# heatmap dùng một hue xanh sáng -> đậm, lưới và trục nhạt, chữ không mang màu series
from matplotlib.colors import LinearSegmentedColormap

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a"]          # 3 slot đầu: phân biệt được cả với người mù màu
MODEL_COLOR = {m: PALETTE[i] for i, m in enumerate(REC_MODELS)}
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
INK, MUTED, GRID, SURFACE = "#1f2328", "#59636e", "#d8dee4", "#fcfcfb"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
                     "axes.titlesize": 11, "axes.labelcolor": MUTED, "text.color": INK})
pct = plt.FuncFormatter(lambda v, _: f"{v * 100:.4g}%")

def style_axes(ax, grid="y", percent_axis="y"):
    ax.grid(axis=grid, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED, length=0)
    if percent_axis:
        getattr(ax, f"{percent_axis}axis").set_major_formatter(pct)

def end_labels(ax, points, min_gap=0.06):
    """Nhãn giá trị ở cuối mỗi đường; đẩy các nhãn gần nhau ra để không chồng."""
    lo, hi = ax.get_ylim()
    gap, last = (hi - lo) * min_gap, -np.inf
    for x, y in sorted(points, key=lambda p: p[1]):
        pos = max(y, last + gap)
        ax.annotate(f"{y:.0%}", xy=(x, y), xytext=(x + 0.12, pos), textcoords="data", va="center", fontsize=9)
        last = pos

def model_legend(fig, **kw):
    handles = [plt.Line2D([], [], color=MODEL_COLOR[m], linewidth=2, marker="o", markersize=8, markeredgecolor="white") for m in REC_MODELS]
    fig.legend(handles, REC_MODELS, frameon=False, ncol=len(REC_MODELS), loc="lower center", bbox_to_anchor=(0.5, -0.02), **kw)

## 2. Tìm ảnh và ground truth

Quét đệ quy `DATA_DIR` lấy mọi ảnh khớp `IMAGE_PATTERN`. Ground truth tra theo đường dẫn ảnh từ mọi `ground_truth.json` tìm được. Mỗi dòng của `items` là một trường trên một ảnh. Bbox của ảnh nhiễu đã được generator biến đổi theo phép xoay/phối cảnh.

In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Không thấy {DATA_DIR.resolve()}. Xem mục 1 để đặt dữ liệu.")

images = sorted(p.as_posix() for p in DATA_DIR.rglob("*") if IMAGE_PATTERN.match(p.name))[:MAX_IMAGES]

def index_ground_truth(data_dir: Path) -> dict:
    """Đường dẫn ảnh -> (thông tin giấy tờ, level, page, bbox theo tên trường)."""
    index = {}
    for gt_path in data_dir.rglob("ground_truth.json"):
        gt = json.loads(gt_path.read_text(encoding="utf-8"))
        for doc in gt["documents"]:
            clean_boxes = {f["name"]: f.get("bbox", []) for f in doc["fields"]}
            pages = [("clean", i + 1, p, clean_boxes) for i, p in enumerate(doc["files"]["pages"])]
            pages += [(n["level"], n["page"], n["path"], n["bboxes"]) for n in doc["files"].get("noisy", [])]
            for level, page, rel, boxes in pages:
                index[(gt_path.parent / rel).as_posix()] = (gt["dossier_id"], doc, level, page, boxes)
    return index

def build_items(images, gt_index) -> pd.DataFrame:
    rows = []
    for im in images:
        if im not in gt_index:
            continue
        dossier, doc, level, page, boxes = gt_index[im]
        for f in doc["fields"]:
            if f.get("fill") == "checkbox" or not f["text"].strip():
                continue
            for k, b in enumerate(bb for bb in boxes.get(f["name"], []) if bb.get("page", 1) == page):
                rows.append(dict(
                    dossier=dossier, doc_id=doc["doc_id"], doc_type=doc["doc_type"], level=level, image=im,
                    field=f["name"], field_type=f["type"], fill=f.get("fill", "printed"),
                    occurrence=k, truth=f["text"], x=b["x"], y=b["y"], w=b["w"], h=b["h"],
                ))
    return pd.DataFrame(rows)

gt_index = index_ground_truth(DATA_DIR)
items = build_items(images, gt_index)
missing = [im for im in images if im not in gt_index]
if missing:
    print(f"Bỏ qua {len(missing)} ảnh không có trong ground_truth.json nào, ví dụ {missing[0]}")
    images = [im for im in images if im in gt_index]
if items.empty:
    raise RuntimeError(f"Không tìm thấy ảnh generator trong {DATA_DIR.resolve()}")
print(f"{len(images)} ảnh, {items.dossier.nunique()} hồ sơ, {len(items)} trường cần đọc")
items["level"] = pd.Categorical(items["level"], LEVELS, ordered=True)
items.groupby(["doc_type", "fill"]).field.unique().to_frame("fields")

Xem thử một ảnh nhiễu nặng cạnh bản sạch, kèm bbox ground truth:

In [ ]:
def draw_boxes(image_path, df, color=(214, 89, 38)):
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    for r in df.itertuples():
        cv2.rectangle(img, (int(r.x), int(r.y)), (int(r.x + r.w), int(r.y + r.h)), color, 3)
    return img

sample = items[items.dossier == items.dossier.iloc[0]]
first_doc = sample.doc_type.iloc[0]
fig, axes = plt.subplots(1, 2, figsize=(14, 10))
for ax, level in zip(axes, ["clean", "heavy"]):
    s = sample[(sample.doc_type == first_doc) & (sample.level == level)]
    if len(s):
        ax.imshow(draw_boxes(s.image.iloc[0], s))
    ax.set_title(f"{first_doc} - {level}")
    ax.axis("off")
plt.tight_layout()

## 3. Chạy OCR cả trang

Kết quả mỗi ảnh được cache thành JSON trong `outputs/paddleocr/<model>/`. Xóa thư mục cache để chạy lại.

In [ ]:
from paddleocr import PaddleOCR

def cache_path(rec_model, image):
    return CACHE_DIR / rec_model / Path(image).relative_to(DATA_DIR).with_suffix(".json")

def run_pipeline(rec_model, image_list):
    todo = [im for im in image_list if not cache_path(rec_model, im).exists()]
    print(f"{rec_model}: {len(image_list) - len(todo)} ảnh đã cache, chạy {len(todo)} ảnh")
    if not todo:
        return
    ocr = PaddleOCR(
        text_detection_model_name=DET_MODEL,
        text_recognition_model_name=rec_model,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        **PADDLE_KWARGS,
    )
    for i, im in enumerate(todo, 1):
        t0 = time.perf_counter()
        r = ocr.predict(im)[0]
        out = dict(
            seconds=time.perf_counter() - t0,
            lines=[dict(text=t, score=float(s), box=[int(v) for v in b])
                   for t, s, b in zip(r["rec_texts"], r["rec_scores"], r["rec_boxes"])],
        )
        cp = cache_path(rec_model, im)
        cp.parent.mkdir(parents=True, exist_ok=True)
        cp.write_text(json.dumps(out, ensure_ascii=False), encoding="utf-8")
        if i % 10 == 0 or i == len(todo):
            print(f"  {i}/{len(todo)} ({out['seconds']:.1f}s/ảnh)")
    del ocr

for rec_model in REC_MODELS:
    run_pipeline(rec_model, images)

## 4. Chấm điểm end-to-end

In [ ]:
def norm(s: str) -> str:
    return " ".join(unicodedata.normalize("NFC", s).split())

def strip_marks(s: str) -> str:
    s = unicodedata.normalize("NFD", s.replace("đ", "d").replace("Đ", "D"))
    return "".join(c for c in s if unicodedata.category(c) != "Mn").casefold()

def semiglobal_distance(gt: str, hyp: str) -> int:
    """Số phép sửa ít nhất để gt thành một đoạn con liên tiếp của hyp. Dùng được cho chuỗi ký tự hoặc danh sách từ."""
    if not gt:
        return 0
    prev = [0] * (len(hyp) + 1)
    for i, g in enumerate(gt, 1):
        cur = [i] + [0] * len(hyp)
        for j, h in enumerate(hyp, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (g != h))
        prev = cur
    return min(prev)

def overlap_lines(lines, x, y, w, h, min_cover=0.5):
    """Dòng OCR mà phần giao chiếm >= min_cover diện tích của bên nhỏ hơn (trường hoặc dòng)."""
    hits = []
    for ln in lines:
        x0, y0, x1, y1 = ln["box"]
        iw = min(x + w, x1) - max(x, x0)
        ih = min(y + h, y1) - max(y, y0)
        if iw <= 0 or ih <= 0:
            continue
        if iw * ih / min(w * h, (x1 - x0) * (y1 - y0)) >= min_cover:
            hits.append(ln)
    return sorted(hits, key=lambda l: ((l["box"][1] + l["box"][3]) / 2, l["box"][0]))

def score_e2e(rec_model):
    ocr_cache = {im: json.loads(cache_path(rec_model, im).read_text(encoding="utf-8")) for im in items.image.unique()}
    out = []
    for r in items.itertuples():
        hits = overlap_lines(ocr_cache[r.image]["lines"], r.x, r.y, r.w, r.h)
        hyp = norm(" ".join(l["text"] for l in hits))
        gt = norm(r.truth)
        out.append(dict(
            detected=bool(hits), hyp=hyp,
            score=float(np.mean([l["score"] for l in hits])) if hits else 0.0,
            cer=semiglobal_distance(gt, hyp) / len(gt),
            cer_nomark=semiglobal_distance(strip_marks(gt), strip_marks(hyp)) / len(gt),
            wer=semiglobal_distance(gt.split(), hyp.split()) / len(gt.split()),
        ))
    return items.assign(model=rec_model, **pd.DataFrame(out, index=items.index))

e2e = pd.concat([score_e2e(m) for m in REC_MODELS], ignore_index=True)
e2e["exact"] = e2e.cer == 0
e2e["cer"] = e2e.cer.clip(upper=1)
e2e["cer_nomark"] = e2e.cer_nomark.clip(upper=1)
e2e["wer"] = e2e.wer.clip(upper=1)

seconds = pd.DataFrame(
    [dict(model=m, seconds=json.loads(cache_path(m, im).read_text())["seconds"]) for m in REC_MODELS for im in images]
).groupby("model").seconds.mean().rename("giây/ảnh")

summary = (e2e.groupby("model")
           .agg(**{"CER": ("cer", "mean"), "WER": ("wer", "mean"), "Field Accuracy": ("exact", "mean"),
                   "CER bỏ dấu": ("cer_nomark", "mean"), "Detection recall": ("detected", "mean")})
           .join(seconds).loc[REC_MODELS])
summary.style.format({c: "{:.1%}" for c in summary.columns if c != "giây/ảnh"} | {"giây/ảnh": "{:.1f}"})

### CER theo mức nhiễu, tách chữ viết tay và chữ in

In [ ]:
def plot_by_level(df, metric, title):
    fills = [f for f in ["handwritten", "printed"] if f in set(df.fill)]
    agg = df.groupby(["fill", "model", "level"], observed=True)[metric].mean()
    fig, axes = plt.subplots(1, len(fills), figsize=(6 * len(fills), 4), sharey=True, squeeze=False)
    ymax = max(agg.max() * 1.15, 0.05)
    for ax, fill in zip(axes[0], fills):
        ends = []
        for m in REC_MODELS:
            s = agg.loc[(fill, m)].reindex(LEVELS)
            ax.plot(LEVELS, s.values, color=MODEL_COLOR[m], linewidth=2, marker="o", markersize=8,
                    markeredgecolor="white", markeredgewidth=2, label=m)
            ends.append((len(LEVELS) - 1, s.values[-1]))
        ax.set_title(fill, loc="left")
        ax.set_ylim(0, ymax)
        ax.set_xlim(-0.2, len(LEVELS) - 0.4)
        end_labels(ax, ends)
        style_axes(ax)
    axes[0][0].set_ylabel("CER")
    fig.suptitle(title, x=0.01, ha="left")
    plt.tight_layout(rect=(0, 0.06, 1, 1))
    model_legend(fig)

plot_by_level(e2e, "cer", "CER end-to-end (thấp hơn là tốt hơn)")

In [ ]:
(e2e.pivot_table(index=["fill", "level"], columns="model", values=["cer", "cer_nomark"], aggfunc="mean", observed=True)
    .reindex(columns=REC_MODELS, level=1)
    .style.format("{:.1%}").background_gradient(cmap=SEQ, vmin=0, vmax=0.5))

### CER theo trường

Trường nào khó nhất (gộp mọi mức nhiễu):

In [ ]:
(e2e.pivot_table(index=["doc_type", "field", "fill"], columns="model", values="cer", aggfunc="mean", observed=True)
    [REC_MODELS].sort_values(REC_MODELS[0], ascending=False)
    .style.format("{:.1%}").background_gradient(cmap=SEQ, vmin=0, vmax=0.5))

## 5. Recognition-only trên vùng cắt theo bbox ground truth

Cắt đúng bbox trường (nới 6px) rồi cho model nhận dạng đọc trực tiếp. Nếu CER ở đây thấp mà end-to-end cao thì lỗi nằm ở detection hoặc ở cách ghép dòng.

In [ ]:
from paddleocr import TextRecognition

PAD = 6

def crop(img, r):
    H, W = img.shape[:2]
    x0, y0 = max(int(r.x) - PAD, 0), max(int(r.y) - PAD, 0)
    x1, y1 = min(int(r.x + r.w) + PAD, W), min(int(r.y + r.h) + PAD, H)
    return img[y0:y1, x0:x1]

def run_rec_only(rec_model):
    cp = CACHE_DIR / rec_model / "_rec_only.json"
    cache = json.loads(cp.read_text(encoding="utf-8")) if cp.exists() else {}
    keys = [f"{Path(r.image).relative_to(DATA_DIR).as_posix()}|{r.field}|{r.occurrence}" for r in items.itertuples()]
    todo = [(k, r) for k, r in zip(keys, items.itertuples()) if k not in cache]
    print(f"{rec_model}: chạy {len(todo)} vùng cắt")
    if todo:
        model = TextRecognition(model_name=rec_model, **PADDLE_KWARGS)
        img_cache = {}
        for k, r in todo:
            img = img_cache.setdefault(r.image, cv2.imread(r.image))
            res = model.predict(crop(img, r))[0]
            cache[k] = dict(text=res["rec_text"], score=float(res["rec_score"]))
        cp.parent.mkdir(parents=True, exist_ok=True)
        cp.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")
        del model
    hyp = [norm(cache[k]["text"]) for k in keys]
    gt = [norm(g) for g in items.truth]
    return items.assign(
        model=rec_model, hyp=hyp, score=[cache[k]["score"] for k in keys],
        cer=[min(Levenshtein.distance(g, h) / len(g), 1) for g, h in zip(gt, hyp)],
        cer_nomark=[min(Levenshtein.distance(strip_marks(g), strip_marks(h)) / len(g), 1) for g, h in zip(gt, hyp)],
        wer=[min(Levenshtein.distance(g.split(), h.split()) / len(g.split()), 1) for g, h in zip(gt, hyp)],
    )

rec = pd.concat([run_rec_only(m) for m in REC_MODELS], ignore_index=True)
rec["exact"] = rec.cer == 0

agg = {"CER": ("cer", "mean"), "WER": ("wer", "mean"), "Field Accuracy": ("exact", "mean"), "CER bỏ dấu": ("cer_nomark", "mean")}
both = pd.concat([
    e2e.groupby("model").agg(**agg).assign(che_do="end-to-end"),
    rec.groupby("model").agg(**agg).assign(che_do="rec-only"),
]).reset_index().set_index(["model", "che_do"]).loc[REC_MODELS]
both.style.format("{:.1%}")

In [ ]:
plot_by_level(rec, "cer", "CER recognition-only trên vùng cắt")

## 6. Biểu đồ độ tương thích model – dữ liệu

**Bốn chỉ số chính:**

| Chỉ số | Cách tính | Tốt khi |
|---|---|---|
| CER | Số ký tự sai / số ký tự ground truth, trung bình theo trường | Thấp |
| WER | Số từ sai / số từ ground truth, trung bình theo trường. Sai một dấu là sai cả từ | Thấp |
| Field Accuracy | Tỷ lệ trường đọc đúng hoàn toàn (CER = 0) | Cao |
| Document Accuracy | Tỷ lệ giấy tờ (một giấy tờ ở một mức nhiễu) có **mọi** trường đọc đúng | Cao |

Document Accuracy là chỉ số khắt khe nhất, gần với câu hỏi thực tế: giấy tờ này có qua được mà không cần người sửa không.

**Chỉ số phụ:**

| Chỉ số | Ý nghĩa | Tốt khi |
|---|---|---|
| CER bỏ dấu | CER sau khi bỏ dấu, chữ hoa | Thấp |
| Detection recall | Tỷ lệ trường có dòng OCR chồng lên | Cao |
| Độ suy giảm | CER(heavy) − CER(clean): độ nhạy với nhiễu | Thấp |
| ECE | Sai lệch giữa điểm tự tin của model và Field Accuracy thực tế | Thấp |

In [ ]:
MAIN = [("CER", False), ("WER", False), ("Field Accuracy", True), ("Document Accuracy", True)]

def doc_level(df):
    """Một dòng cho mỗi giấy tờ ở một mức nhiễu: đúng khi mọi trường đều đúng."""
    return (df.groupby(["model", "dossier", "doc_id", "doc_type", "level"], observed=True)
              .exact.all().rename("doc_exact").reset_index())

def main_metrics(df, by=()):
    keys = ["model", *by]
    fields = df.groupby(keys, observed=True).agg(**{"CER": ("cer", "mean"), "WER": ("wer", "mean"),
                                                    "Field Accuracy": ("exact", "mean")})
    docs = doc_level(df).groupby(keys, observed=True).doc_exact.mean().rename("Document Accuracy")
    return fields.join(docs)

def scorecard(table, title, metrics):
    """table: index = model, mỗi cột một chỉ số. Mỗi chỉ số một ô, thanh ngang theo model."""
    fig, axes = plt.subplots(1, len(metrics), figsize=(3.4 * len(metrics), 0.55 * len(REC_MODELS) + 1.4), sharey=True)
    y = np.arange(len(REC_MODELS))
    for i, (ax, (col, higher_better)) in enumerate(zip(np.atleast_1d(axes), metrics)):
        vals = table[col].reindex(REC_MODELS).values
        ax.barh(y, vals, height=0.62, color=[MODEL_COLOR[m] for m in REC_MODELS], edgecolor=SURFACE, linewidth=2)
        for yi, v in zip(y, vals):
            ax.text(v, yi, f" {v:.1%}", va="center", fontsize=9, color=INK)
        ax.set_xlim(0, 1.18 if higher_better else max(np.nanmax(vals) * 1.35, 0.05))
        ax.set_title(f"{col}\n({'cao' if higher_better else 'thấp'} là tốt)", loc="left", fontsize=10)
        style_axes(ax, grid="x", percent_axis="x")
        ax.tick_params(labelleft=(i == 0))
    axes[0].set_yticks(y, REC_MODELS)
    axes[0].invert_yaxis()
    fig.suptitle(title, x=0.01, ha="left")
    plt.tight_layout()

main_e2e, main_rec = main_metrics(e2e), main_metrics(rec)

### 6.1. Bốn chỉ số chính

In [ ]:
scorecard(main_e2e, "End-to-end (detection + recognition, cả trang)", MAIN)
scorecard(main_rec, "Recognition-only (vùng cắt theo bbox ground truth)", MAIN)

### 6.2. Bốn chỉ số theo mức nhiễu

End-to-end. Đường dốc càng mạnh thì model càng nhạy với chất lượng ảnh. Document Accuracy thường rơi nhanh nhất vì chỉ cần một trường sai là cả giấy tờ trượt.

In [ ]:
by_level = main_metrics(e2e, by=["level"])
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, (col, higher_better) in zip(axes, MAIN):
    ends = []
    for m in REC_MODELS:
        s = by_level.loc[m, col].reindex(LEVELS)
        ax.plot(LEVELS, s.values, color=MODEL_COLOR[m], linewidth=2, marker="o", markersize=8,
                markeredgecolor="white", markeredgewidth=2)
        ends.append((len(LEVELS) - 1, s.values[-1]))
    ax.set_ylim(0, 1.05 if higher_better else max(by_level[col].max() * 1.2, 0.05))
    ax.set_xlim(-0.2, len(LEVELS) - 0.4)
    end_labels(ax, ends)
    ax.set_title(f"{col} ({'cao' if higher_better else 'thấp'} là tốt)", loc="left")
    style_axes(ax)
plt.tight_layout(rect=(0, 0.08, 1, 1))
model_legend(fig)

### 6.3. Document Accuracy theo loại giấy tờ

Mỗi ô là tỷ lệ giấy tờ loại đó, ở mức nhiễu đó, được đọc đúng mọi trường. Ô nhạt là loại giấy tờ model chưa hợp (màu đậm = tỷ lệ cao = tốt).

In [ ]:
docs = doc_level(e2e)
doc_types = sorted(docs.doc_type.unique())
fig, axes = plt.subplots(1, len(REC_MODELS), figsize=(3.2 * len(REC_MODELS) + 3, 0.45 * len(doc_types) + 1.8),
                         sharey=True, layout="constrained")
for i, (ax, m) in enumerate(zip(np.atleast_1d(axes), REC_MODELS)):
    grid = (docs[docs.model == m].groupby(["doc_type", "level"], observed=True).doc_exact.mean()
            .unstack("level").reindex(index=doc_types, columns=LEVELS))
    im = ax.imshow(grid.values, cmap=SEQ, vmin=0, vmax=1, aspect="auto")
    for (r, c), v in np.ndenumerate(grid.values):
        if not np.isnan(v):
            ax.text(c, r, f"{v:.0%}", ha="center", va="center", fontsize=8, color="white" if v > 0.55 else INK)
    ax.set_xticks(range(len(LEVELS)), LEVELS)
    ax.tick_params(colors=MUTED, length=0, labelleft=(i == 0))
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_title(m, loc="left")
axes[0].set_yticks(range(len(doc_types)), doc_types)
fig.colorbar(im, ax=axes, format=pct, shrink=0.8, label="Document Accuracy").outline.set_visible(False)
fig.suptitle("Document Accuracy theo loại giấy tờ và mức nhiễu", x=0.01, ha="left")

### 6.4. Chỉ số phụ và nguồn lỗi

CER tách làm hai phần: **lỗi mặt chữ** (còn sai kể cả khi bỏ dấu) và **lỗi dấu** (phần chênh giữa CER và CER bỏ dấu). Phần lỗi dấu lớn nghĩa là model đọc đúng chữ cái nhưng không hợp với tiếng Việt.

In [ ]:
aux = pd.DataFrame({"CER bỏ dấu": e2e.groupby("model").cer_nomark.mean(),
                    "Detection recall": e2e.groupby("model").detected.mean()})
scorecard(aux, "Chỉ số phụ, end-to-end", [("CER bỏ dấu", False), ("Detection recall", True)])

ERR_COLORS = {"Lỗi mặt chữ": "#184f95", "Lỗi dấu": "#86b6ef"}
rows = []
for mode, df in [("end-to-end", e2e), ("rec-only", rec)]:
    g = df.groupby("model")[["cer", "cer_nomark"]].mean()
    for m in REC_MODELS:
        rows.append(dict(label=f"{m}\n{mode}", **{"Lỗi mặt chữ": g.loc[m, "cer_nomark"],
                                                   "Lỗi dấu": max(g.loc[m, "cer"] - g.loc[m, "cer_nomark"], 0)}))
err = pd.DataFrame(rows).set_index("label")

fig, ax = plt.subplots(figsize=(9, 0.5 * len(err) + 1.2))
left = np.zeros(len(err))
for name, color in ERR_COLORS.items():
    ax.barh(err.index, err[name], left=left, height=0.62, color=color, edgecolor=SURFACE, linewidth=2, label=name)
    left += err[name].values
for yi, total in enumerate(left):
    ax.text(total, yi, f" {total:.1%}", va="center", fontsize=9, color=INK)
ax.invert_yaxis()
ax.set_xlim(0, max(left.max() * 1.2, 0.05))
style_axes(ax, grid="x", percent_axis="x")
ax.set_xlabel("CER")
ax.legend(frameon=False, ncol=2, loc="lower right", bbox_to_anchor=(1, 1))
ax.set_title("CER tách theo nguồn lỗi", loc="left")
plt.tight_layout()

### 6.5. Bản đồ nhiệt CER: trường × mức nhiễu

CER end-to-end của từng trường theo từng mức nhiễu, mỗi model một ô, cùng thang màu. Ô đậm là chỗ model không hợp với dữ liệu. Chỉ vẽ `TOP_FIELDS` trường khó nhất, bảng đầy đủ ở mục 4.

In [ ]:
TOP_FIELDS = 20
fields = e2e.groupby(["doc_type", "field"]).cer.mean().sort_values(ascending=False).index[:TOP_FIELDS]
field_labels = [f"{d} / {f}" for d, f in fields]
vmax = max(e2e.groupby(["model", "doc_type", "field", "level"], observed=True).cer.mean().max(), 0.05)

fig, axes = plt.subplots(1, len(REC_MODELS), figsize=(3.2 * len(REC_MODELS) + 3, 0.34 * len(fields) + 1.8),
                         sharey=True, layout="constrained")
for i, (ax, m) in enumerate(zip(np.atleast_1d(axes), REC_MODELS)):
    grid = (e2e[e2e.model == m].groupby(["doc_type", "field", "level"], observed=True).cer.mean()
            .unstack("level").reindex(index=fields, columns=LEVELS))
    im = ax.imshow(grid.values, cmap=SEQ, vmin=0, vmax=vmax, aspect="auto")
    for (r, c), v in np.ndenumerate(grid.values):
        if not np.isnan(v):
            ax.text(c, r, f"{v:.0%}", ha="center", va="center", fontsize=8, color="white" if v > vmax * 0.55 else INK)
    ax.set_xticks(range(len(LEVELS)), LEVELS)
    ax.tick_params(colors=MUTED, length=0, labelleft=(i == 0))
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_title(m, loc="left")
axes[0].set_yticks(range(len(fields)), field_labels)
fig.colorbar(im, ax=axes, format=pct, shrink=0.6, label="CER").outline.set_visible(False)
fig.suptitle(f"CER theo trường và mức nhiễu ({len(fields)} trường khó nhất)", x=0.01, ha="left")

### 6.6. Phân bố CER và WER theo trường

Đường cho biết tỷ lệ trường có lỗi ≤ x. Điểm ở x = 0 của CER chính là Field Accuracy. Đường nằm cao và dốc sớm nghĩa là phần lớn trường chỉ sai ít, hậu xử lý dễ sửa.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, col in zip(axes, ["cer", "wer"]):
    for m in REC_MODELS:
        x = np.sort(e2e[e2e.model == m][col].values)
        ax.step(np.r_[0, x], np.r_[0, np.arange(1, len(x) + 1) / len(x)], where="post", color=MODEL_COLOR[m], linewidth=2)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.set_title(f"Tỷ lệ trường có {col.upper()} ≤ x", loc="left")
    ax.set_xlabel(col.upper())
    style_axes(ax, grid="both")
    ax.xaxis.set_major_formatter(pct)
plt.tight_layout(rect=(0, 0.07, 1, 1))
model_legend(fig)

### 6.7. Hiệu chỉnh điểm tự tin (reliability diagram)

Chia các vùng cắt theo `rec_score` của model, so điểm tự tin trung bình với Field Accuracy thực tế. Model hiệu chỉnh tốt nằm sát đường chéo. Nằm dưới đường chéo là **tự tin quá mức**: không thể dùng ngưỡng điểm để quyết định trường nào cần người kiểm tra lại.

In [ ]:
BINS = [0, 0.5, 0.7, 0.8, 0.9, 0.95, 0.98, 1.0001]

def calibration(df):
    b = pd.cut(df.score, BINS, right=False)
    g = df.groupby(b, observed=True).agg(conf=("score", "mean"), acc=("exact", "mean"), n=("exact", "size"))
    ece = float((g.n / g.n.sum() * (g.acc - g.conf).abs()).sum())
    return g, ece

fig, ax = plt.subplots(figsize=(6, 5.2))
ax.plot([0, 1], [0, 1], color=MUTED, linewidth=1, linestyle="--")
ax.text(0.62, 0.66, "hiệu chỉnh hoàn hảo", color=MUTED, fontsize=8, rotation=45, ha="center")
ece_by_model = {}
for m in REC_MODELS:
    g, ece = calibration(rec[rec.model == m])
    ece_by_model[m] = ece
    ax.plot(g.conf, g.acc, color=MODEL_COLOR[m], linewidth=2, marker="o", markersize=8,
            markeredgecolor="white", markeredgewidth=2, label=f"{m}  (ECE {ece:.1%})")
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.set_aspect("equal")
ax.set_xlabel("Điểm tự tin trung bình (rec_score)")
ax.set_ylabel("Field Accuracy thực tế")
ax.set_title("Reliability diagram, recognition-only", loc="left")
style_axes(ax, grid="both")
ax.xaxis.set_major_formatter(pct)
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(0, -0.14), ncol=1)
plt.tight_layout()

### 6.8. Đánh đổi tốc độ và độ chính xác

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
cer_e2e = main_e2e["CER"]
for m in REC_MODELS:
    ax.scatter(seconds[m], cer_e2e[m], s=90, color=MODEL_COLOR[m], edgecolor="white", linewidth=2, zorder=3)
    ax.annotate(m, (seconds[m], cer_e2e[m]), xytext=(9, 0), textcoords="offset points", va="center", fontsize=9)
ax.set_xlim(0, seconds.max() * 1.5)
ax.set_ylim(0, max(cer_e2e.max() * 1.3, 0.05))
ax.set_xlabel(f"Giây/ảnh ({PADDLE_KWARGS['device']})")
ax.set_ylabel("CER end-to-end")
ax.set_title("Góc dưới trái là tốt nhất: nhanh và ít lỗi", loc="left")
style_axes(ax, grid="both")
plt.tight_layout()

### 6.9. Bảng tổng hợp hệ số

In [ ]:
cer_lv = e2e.groupby(["model", "level"], observed=True).cer.mean().unstack("level")
coef = main_e2e.join(main_rec.add_suffix(" (rec-only)")).assign(**{
    "CER bỏ dấu": e2e.groupby("model").cer_nomark.mean(),
    "Detection recall": e2e.groupby("model").detected.mean(),
    "Độ suy giảm": cer_lv.get("heavy") - cer_lv.get("clean"),
    "ECE": pd.Series(ece_by_model),
    "Giây/ảnh": seconds,
}).loc[REC_MODELS]

higher_better = [c for c in coef.columns if "Accuracy" in c or c == "Detection recall"]
lower_better = [c for c in coef.columns if c not in higher_better]
(coef.style
    .format({c: "{:.1%}" for c in coef.columns if c != "Giây/ảnh"} | {"Giây/ảnh": "{:.1f}"})
    .highlight_min(subset=lower_better, props="font-weight: bold")
    .highlight_max(subset=higher_better, props="font-weight: bold"))

## 7. Ký tự hay bị đọc sai

Đếm trên kết quả recognition-only: mỗi ký tự ground truth bị xóa hoặc thay bằng ký tự khác. Tỷ lệ = số lần sai / số lần xuất hiện.

In [ ]:
from collections import Counter

def char_errors(df):
    seen, wrong, became = Counter(), Counter(), {}
    for g, h in zip(df.truth.map(norm), df.hyp):
        seen.update(g)
        for op in Levenshtein.editops(g, h):
            if op.tag in ("delete", "replace"):
                c = g[op.src_pos]
                wrong[c] += 1
                became.setdefault(c, Counter())[h[op.dest_pos] if op.tag == "replace" else "∅"] += 1
    rows = [dict(char=c, xuat_hien=seen[c], sai=wrong[c], ti_le=wrong[c] / seen[c],
                 thanh=", ".join(f"{k}×{v}" for k, v in became[c].most_common(3)))
            for c in wrong if c != " "]
    return pd.DataFrame(rows).sort_values(["ti_le", "sai"], ascending=False)

for m in REC_MODELS:
    print(f"\n=== {m} (chỉ ảnh sạch) ===")
    display(char_errors(rec[(rec.model == m) & (rec.level == "clean")]).head(15).style.format({"ti_le": "{:.0%}"}).hide(axis="index"))

## 8. Xem các ca sai nặng nhất

In [ ]:
def show_worst(df, model, n=12, level=None):
    s = df[(df.model == model) & (df.cer > 0)]
    if level:
        s = s[s.level == level]
    s = s.sort_values("cer", ascending=False).head(n)
    if s.empty:
        print("Không có ca sai")
        return
    fig, axes = plt.subplots(len(s), 1, figsize=(10, 1.1 * len(s)))
    axes = np.atleast_1d(axes)
    for ax, r in zip(axes, s.itertuples()):
        img = cv2.cvtColor(crop(cv2.imread(r.image), r), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"[{r.level}/{r.fill}] {r.field}  CER {r.cer:.0%}\nGT : {r.truth}\nOCR: {r.hyp}",
                     loc="left", fontsize=9, color=INK, family="monospace")
    plt.tight_layout()

show_worst(rec, REC_MODELS[0], level="clean")

In [ ]:
show_worst(rec, REC_MODELS[0], level="heavy")

## 9. Ghi chú đọc kết quả

- Model nào có CER bỏ dấu thấp nhưng CER cao: đọc được mặt chữ, hỏng ở dấu. Với tiếng Việt, cần model có từ điển đủ ký tự có dấu chồng.
- CER end-to-end cao hơn rec-only nhiều: kiểm tra detection (dòng bị cắt đôi, dòng dính vào nhau) và ngưỡng `min_cover` trong `overlap_lines`.
- Chữ viết tay ở golden set là font viết tay, dễ hơn chữ viết tay thật. Cần kiểm lại trên `real/` trước khi chốt model.